# HDF MODIS files 

This script contains an overview of HDF files used in this EVI analysis. Each file downloaded by earthdata is then converted in .nc file and finally all files are converted in one unique file contains all timeseries for each pixel worldwide.

### 📁 What’s Inside a MOD13C1 HDF File?

- NDVI

- EVI

- Pixel Reliability

- VI Quality

- composite_day_of_the_year

Each of these layers is a separate data grid, usually organized in latitude-longitude format, and can be accessed individually.

In [ ]:
# needed libraries
import os
from pyhdf.SD import SD, SDC
import numpy as np
from datetime import datetime
import xarray as xr
import re
import pandas as pd

In [ ]:
# Just as an example and to see what is inside the HDF files downloaded from NASA Earthdata, consider a single file and inspect it.
# inspect one file to see what variables are available
file_path = "MOD13C2.A2023001.h21v10.006.2023021153014.hdf" # replace with your file path that you downloaded by NASA Earthdata
hdf = SD(file_path, SDC.READ)

# Print all datasets to see what is inside the HDF file, all the layers:
print("Datasets in the file:")
for idx, sds_name in enumerate(hdf.datasets().keys()):
    print(f"{idx+1}. {sds_name}")

Datasets in the file:
1. CMG 0.05 Deg 16 days NDVI
2. CMG 0.05 Deg 16 days EVI
3. CMG 0.05 Deg 16 days VI Quality
4. CMG 0.05 Deg 16 days red reflectance
5. CMG 0.05 Deg 16 days NIR reflectance
6. CMG 0.05 Deg 16 days blue reflectance
7. CMG 0.05 Deg 16 days MIR reflectance
8. CMG 0.05 Deg 16 days Avg sun zen angle
9. CMG 0.05 Deg 16 days NDVI std dev
10. CMG 0.05 Deg 16 days EVI std dev
11. CMG 0.05 Deg 16 days #1km pix used
12. CMG 0.05 Deg 16 days #1km pix +-30deg VZ
13. CMG 0.05 Deg 16 days pixel reliability


In [ ]:
# Folder with HDF files
folder = "./earthdata/MOD13C1_16days" # replace with your actual folder path
hdf_files = sorted([f for f in os.listdir(folder) if f.endswith('.hdf')]) # get list of HDF files, sorted
hdf_paths = [os.path.join(folder, f) for f in hdf_files]

In [ ]:
# File names follows this pattern: MOD13C1.A<YYYY><DOY>.061.<ProductionDate>.hdf, this patterns will be used to extract date information
# extract dates from all files
all_dates = []

for i, file in enumerate(sorted(hdf_files)):
    path = os.path.join(folder, file)
    hdf  = SD(path, SDC.READ)

    match = re.search(r'A(\d{4})(\d{3})', file)
    date  = datetime.strptime(f"{match.group(1)}{match.group(2)}", "%Y%j")

    all_dates.append(date)  

# save all dates to a pandas df and then a CSV file
dates_df = pd.DataFrame({"date": all_dates})
dates_df.to_csv("all_evi_dates.csv", index=False)

Now we proceed by converting the files to netcdf and store them temporarly until we create a merged file (this latter file will be quite big, in the orders of hundreds GB). 

In [ ]:
# Folder and file setup
output_dir = "temp_nc_files" # temporary folder to save individual NetCDF files, create if not exists
os.makedirs(output_dir, exist_ok=True)

lat = np.arange(89.975, -90, -0.05)
lon = np.arange(-179.975, 180, 0.05)

# Step 1: Save each time step separately
for i, file in enumerate(sorted(hdf_files)):
    path = os.path.join(folder, file)
    hdf = SD(path, SDC.READ)
    evi = hdf.select('CMG 0.05 Deg 16 days EVI')[:]
    evi = np.where(evi == -3000, np.nan, evi.astype(np.float32) * 0.0001) # scale factor 0.0001 as described from documentation

    # Get date from filename
    match = re.search(r'A(\d{4})(\d{3})', file)
    date = datetime.strptime(f"{match.group(1)}{match.group(2)}", "%Y%j")

    # Create Dataset
    ds = xr.Dataset(
        {"EVI": (["time", "lat", "lon"], evi[np.newaxis, :, :])},
        coords={"time": [date], "lat": lat, "lon": lon}
    )
    
    # Save to individual NetCDF
    out_file = os.path.join(output_dir, f"EVI_{date.strftime('%Y%m%d')}.nc")
    ds.to_netcdf(out_file, format="NETCDF4", engine="netcdf4")


In [ ]:
#Combine all time steps into a single NetCDF file
combined = xr.open_mfdataset(f"{output_dir}/EVI_*.nc", concat_dim="time", combine="nested")
combined.to_netcdf("EVI_time_series_scaled.nc", format="NETCDF4", engine="netcdf4")